<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [4]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
df['ctr'] = df['gsc_clicks']/df['gsc_impressions']
df['ctr'].fillna(0, inplace=True)

In [6]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [7]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [8]:
df['ga4_data_available'] = df['ga4_data_available'].fillna(False)

conditions = [
    df['gsc_data_available'] & df['ga4_data_available'],
    df['gsc_data_available'] & ~df['ga4_data_available'],
    ~df['gsc_data_available'] & df['ga4_data_available'],
    ~df['gsc_data_available'] & ~df['ga4_data_available']
]

tiers = ['Full', 'GSC-only', 'GA4-only', 'unavailable']

df['tier'] = np.select(conditions, tiers, default=None)

In [9]:
df_full_tier = df[df['tier'] == 'Full']

df_full_tier = df_full_tier.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ctr=('ctr', 'mean')
)

In [10]:
df_full_tier = df_full_tier.merge(df_trend[['client_hash_id', 'content_hash_id','gsc_impressions_feb', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [11]:
conditions = [
    df_full_tier['trend_pct'] < -50,
    (df_full_tier['trend_pct'] >= -50) & (df_full_tier['trend_pct'] < -15),
    (df_full_tier['trend_pct'] >= -15) & (df_full_tier['trend_pct'] <= 15),
    (df_full_tier['trend_pct'] > 15) & (df_full_tier['trend_pct'] <= 50),
    df_full_tier['trend_pct'] > 50
]
ranks = ['5-Sharp decline', '4-Mild decline', '3-Flat', '2-Mild growth', '1-Strong growth']
df_full_tier['trend_dir'] = np.select(conditions, ranks, default=None)

In [12]:
display(df_full_tier.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
        'gsc_impressions' : ['mean', 'count'],
        'gsc_clicks' : ['mean', 'count'],
        'ga4_pageviews' : ['mean', 'count'],
        'ga4_sessions' : ['mean', 'count'],
        'ga4_users' : ['mean', 'count'],
        'ga4_engaged_sessions' : ['mean', 'count'],
        'ctr' : ['mean', 'count'],
    }
))

gsc_sum_position        gsc_avg_position         \
                            mean  count             mean  count   
trend_dir                                                         
1-Strong growth      3977.732661   3172        12.064798   3172   
2-Mild growth        6962.328934   3940        11.044186   3940   
3-Flat              14112.335223   8994        11.400757   8994   
4-Mild decline      29316.619630  16037        14.433535  16037   
5-Sharp decline     34057.759308  17458        15.088555  17458   

                gsc_impressions        gsc_clicks        ga4_pageviews         \
                           mean  count       mean  count          mean  count   
trend_dir                                                                       
1-Strong growth      371.755044   3172   2.403531   3172     21.837642   3172   
2-Mild growth        776.021066   3940   4.795939   3940     23.379949   3940   
3-Flat              1347.903602   8994   7.221592   8994     27.404047   8994   
4-Mild decline      1627.273493  16037   6.049448  16037     28.376816  16037   
5-Sharp decline     2134.333085  17458   9.021938  17458     24.133921  17458   

                ga4_sessions         ga4_users        ga4_engaged_sessions  \
                        mean  count       mean  count                 mean   
trend_dir                                                                    
1-Strong growth     20.60971   3172  20.468789   3172             0.243064   
2-Mild growth      21.287563   3940  20.945685   3940             0.372589   
3-Flat             24.079497   8994  23.590838   8994             0.535023   
4-Mild decline     24.685103  16037  24.239633  16037             0.566565   
5-Sharp decline    21.014836  17458  20.353019  17458             0.518043   

                             ctr         
                 count      mean  count  
trend_dir                                
1-Strong growth   3172  0.028362   3172  
2-Mild growth     3940  0.021342   3940  
3-Flat            8994  0.018003   8994  
4-Mild decline   16037  0.015814  16037  
5-Sharp decline  17458  0.019734  17458

In [13]:
"""
gsc_avg_position averages across a page's queries,
but since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),
a page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.
This likely explains some of the scrambled middle-bucket ordering — sum_position,
while confounded by query volume, doesn't suffer from this specific averaging distortion.
"""

"\ngsc_avg_position averages across a page's queries,\nbut since positions 1–10 (page 1) behave very differently from positions beyond 10 (effectively invisible to most searchers),\na page with mixed query performance can average out to a position that doesn't reflect real visibility for any of its queries.\nThis likely explains some of the scrambled middle-bucket ordering — sum_position,\nwhile confounded by query volume, doesn't suffer from this specific averaging distortion.\n"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.